In [5]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
import pickle
from sklearn.model_selection import train_test_split

In [7]:
# 1. Load data
df = pd.read_csv("../data/imdb_preprocessed.csv")
X = df["processed_review"]
y = df["sentiment"].map({"negative": 0, "positive": 1})

**Error Analysis for Logistic Regression**

In [2]:
with open("../model/logistic_regression_model.pkl", "rb") as f:
    logistic_model = pickle.load(f)

print("logistic model loaded successfully!")

logistic model loaded successfully!


In [3]:
with open("../model/tfidf_movie_review.pkl", "rb") as f:
    tfidf = pickle.load(f)

print("tfidf movie review loaded sucessfully!")

tfidf movie review loaded sucessfully!


In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [9]:
X_test_tfidf = tfidf.transform(X_test)

In [10]:
y_pred = logistic_model.predict(X_test_tfidf)

In [11]:
error_df = pd.DataFrame({
    "Review": X_test.values,
    "Actual": y_test.values,
    "Predicted": y_pred
})

In [12]:
errors = error_df[
    error_df["Actual"] != error_df["Predicted"]
].copy()

print(f"Total Incorrect Predictions: {len(errors)}")

errors.head(10)

Total Incorrect Predictions: 1071


,Review,Actual,Predicted
15,not exactly great film admire writer director ...,1,0
21,film start get feeling go something special ac...,0,1
24,despite huge fan fred astaire ginger rogers mo...,0,1
35,well well roeg touch bit nerve not genius cata...,1,0
43,ok let admit not see merchant ivory distributo...,0,1
45,not tell armor kubrickian hitchcockian fellini...,1,0
65,good original nearly achieve greatness cundief...,1,0
106,save grace never riotously funny deliver quite...,1,0
112,financially strapped paramount pull stop stage...,1,0
115,documentary dinosaur undoubtedly fascinating w...,1,0


In [13]:
errors["Actual"] = errors["Actual"].map({
    0: "Negative",
    1: "Positive"
})

errors["Predicted"] = errors["Predicted"].map({
    0: "Negative",
    1: "Positive"
})

errors.head(10)

,Review,Actual,Predicted
15,not exactly great film admire writer director ...,Positive,Negative
21,film start get feeling go something special ac...,Negative,Positive
24,despite huge fan fred astaire ginger rogers mo...,Negative,Positive
35,well well roeg touch bit nerve not genius cata...,Positive,Negative
43,ok let admit not see merchant ivory distributo...,Negative,Positive
45,not tell armor kubrickian hitchcockian fellini...,Positive,Negative
65,good original nearly achieve greatness cundief...,Positive,Negative
106,save grace never riotously funny deliver quite...,Positive,Negative
112,financially strapped paramount pull stop stage...,Positive,Negative
115,documentary dinosaur undoubtedly fascinating w...,Positive,Negative


In [14]:
# Probability of Positive class
y_proba = logistic_model.predict_proba(X_test_tfidf)

# Confidence of the predicted class
errors["Confidence"] = y_proba[
    errors.index,
    errors["Predicted"].map({"Negative": 0, "Positive": 1})
]

errors = errors.sort_values(
    by="Confidence",
    ascending=False
)

errors.head(10)

,Review,Actual,Predicted,Confidence
6096,spoiler ahead really unfortunate movie well pr...,Positive,Negative,0.995206
8795,one best celebrity reality show ever saw see c...,Negative,Positive,0.994803
5774,simply funny movie see long time bad act bad s...,Positive,Negative,0.994431
9026,comanche moon everything go starter simon winc...,Positive,Negative,0.983540
4718,not citizen kane deliver cleavage lot badly ac...,Positive,Negative,0.979180
1450,finally opportunity watch zombie zombie fleshe...,Negative,Positive,0.976240
5305,excerpt nine page memo mr cohn mr welles write...,Positive,Negative,0.975930
7674,honestly expect hate one really check jenna ja...,Positive,Negative,0.967064
7613,helen troy follow story helen outbreak trojan ...,Negative,Positive,0.964445
5367,hollywood long love affair bogus arabian night...,Negative,Positive,0.964009


In [15]:
pd.set_option("display.max_colwidth", None)
errors.head(10)

,Review,Actual,Predicted,Confidence
6096,spoiler ahead really unfortunate movie well produce turn disappointment think full silly clichés basically try hard american guy many spend time jump girlfriend bed make monkey sound marry girl many suddenly go prude nymphos overnight not husband french would really ask someone à la fac know not speak french would not use common word like université live france sort know understand europe love german roommate find pretty insult overall look like movie fund european parliament try hard basically sort difference try tie together not bad thing result best awkward fact ridiculous many clash would not really happen end movie last minute ruin rest not xavier talk erasmus student meet back paris walk run away job freedom end new europe suppose rest bunch people smoke shag day make besides acting pretty horrible not believe judith godrèche role act make look like emanuelle béart much first think xavier ok retrospect think pretty bad really bad technically open credit scene ask paper need really good except sound edit around british sibling soundtrack great form good content pretty horrible,Positive,Negative,0.995206
8795,one best celebrity reality show ever saw see concert see life britney love five episode always surprise britney subject show think people not watch show great person really funny really gentle love fan see love work not give k fed real jerk not seem like britney make second season great show show people britney really go britney best never leave heart,Negative,Positive,0.994803
5774,simply funny movie see long time bad act bad script bad scenery bad costume bad camera work bad special effect stupid find reel laughter not go win oscar get beer friend round not go wrong,Positive,Negative,0.994431
9026,comanche moon everything go starter simon wincer back man name synonymous high quality tv western unfortunately problem moon something even talented director could not solve poor script base lackluster novel forget historical accuracy reader novel tell big travesty moon not even consistent information original lonesome dove masterpiece many wonderful rich moment miniseries even great degree book completely miss moon consider fact viewer moon probably come least sort dove background lack character drive emotional backstory downright painful say moon one instance adapted version could alter make suitable screen example novel moon focus largely comanche credit miniseries try service comanche end give enough viewer get sense miss important part story similarly val kilmer scull lose role either expand kilmer rachel griffith matter actually something role reduce small support part instead kilmer get top billing character leave scratch head appearance completely bizarre final act strange moment throughout film make no sense not read book killer parrot not single scene show u call mccrae anything near amazing texas ranger claim not single one expedition moon dead man walk matter end successfully call mccrae seem blunder way one pointless mission another frankly larry mcmurtry never give job write script praise surround cowboy movie brokeback mountain mcmurtry write good novel although dissension consistency statement never exactly establish scriptwriter production would benefit not bring back director wincer original screenwriter bill wittcliff adapt novel screen witcliff not exactly mountainload material screen write credit no one deny fantastic job whittle original lonesome dove say comanche moon almost brilliant production aid terrific cast unfortunately not give enough steve zahn portrayal gus mccrae rather portrayal robert duvall gus mccrae dead criticize karl urban woodrow call say performance not imitate quiet stoic woodrow original movie say blame mcmurtry mcmurtry one moon novel miniseries turn woodrow socially inept awkward natural leader emotionless character whose line dull whose character motivation clear see portray far adequately sequel still cast sparkle moon degree leave mi

In [16]:
error_summary = errors.groupby(
    ["Actual", "Predicted"]
).size().reset_index(name="Count")

error_summary

,Actual,Predicted,Count
0,Negative,Positive,589
1,Positive,Negative,482


**Error Analysis for SVM**

In [18]:
with open("../model/svm_model.pkl", "rb") as f:
    svm_model = pickle.load(f)

print("Linear SVM model loaded successfully!")

Linear SVM model loaded successfully!


In [20]:
with open("../model/tfidf_movie_review.pkl", "rb") as f:
    tfidf = pickle.load(f)

print("TF-IDF vectorizer loaded successfully!")

TF-IDF vectorizer loaded successfully!


In [21]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [22]:
X_test_tfidf = tfidf.transform(X_test)

print("X_test_tfidf shape:", X_test_tfidf.shape)

X_test_tfidf shape: (9917, 5000)


In [23]:
y_pred_svm = svm_model.predict(X_test_tfidf)

In [24]:
print("Total Test Samples:", len(y_test))
print("Total Incorrect Predictions:", (y_test != y_pred_svm).sum())

Total Test Samples: 9917
Total Incorrect Predictions: 1053


In [25]:
errors = pd.DataFrame({
    "Review": X_test,
    "Actual": y_test,
    "Predicted": y_pred_svm
})

errors = errors[
    errors["Actual"] != errors["Predicted"]
].copy()

print("Total Incorrect Predictions:", len(errors))

errors.head(10)

Total Incorrect Predictions: 1053


,Review,Actual,Predicted
45455,not exactly great film admire writer director try something little different film main theme fate small seemingly insignificant thing greatly change future way remind film slide door though instead focus one random event seemingly random stuff happen repeatedly one help build cute conclusion plus odd bald guy seem understand talk one brief scene like sort omnipotent absolutely no explanation film like two guy fight clock tower hudsucker proxy dvd jacket show audrey tautou capitalize success amelie though one many actor film no one star role pace brisk acting fine conclusion not bad reason not score high character bit uninteresting think movie could perhaps tighten less subplots,1,0
34773,film start get feeling go something special acting camera work undoubtedly good also like character could grow empathise film good atmosphere hint fantasy however film go plot never appear takeoff roll scene scene unable understand connection story could see character occasionally bump reference ship bottle without connection leave unremarkable short story surprise well cannes,0,1
34520,despite huge fan fred astaire ginger rogers movie not year ago first saw follow fleet know song old astaire rogers record yes vinyl know nothing plot unfortunately song catchy ginger rogers character sweet funny not like bake baker try make longtime partner continually sabotage career character not usual humour elan film astaire character worth watch song great solo tap routine ginger rogers,0,1
27110,well well roeg touch bit nerve not genius catalogue various character descent psychosis couple decade soon bad taste suggest redemption even good advice might find bad old catholic church hip thou alternative movie crowd get extra vicious bad still theresa russell character face experience nothing avowedly rationalist outlook explanation unwillingly force deal experience another level spiritual know realm ignorant superstitious sort thing art house cinephiles suppose oh horror find marriage idea might uniquely important commitment affirm seem uncomfortably like divine intervention people find idea prima facie offensive could maybe ask instinctively jump attack mode challenge take seriously idea spiritual dimension life probably not sure film problem notably talia shire delirious hamwork overwrought nun style attire dialogue marie davenport young priest last scene straight spellbound school glib interpretation though hitchcock movie escape similar charge due source wisdom impeccably secular credential freudian psychoanalyst sadly nicolas roeg appear cop critical mauling much even ask question possible answer film present,1,0
33827,ok let admit not see merchant ivory distributor film nor see celebrated work director capacity discuss rain outside analysis film mitigate admittance let begin rain different kind movie not know genre want first pretend romance romance protagonist fall love support character separate support character sometimes united partner movie hero already win heart lover not name henry moore name sajani reside southern india waning day raj british imperial rule henry away london long time fall love married indian maid despite legal marriage child overseas could good often sneak away intimate afternoon child notice word spread sajani husband question involvement moore deny contact moor ask leave area sajani refuse devotion commits suicide please take note event occur opening third film film change tone become crime drama final portion sajani body discover right moore family come visit alleged perpetrator moore english educate assistant k k know master affair keep silent life becomes threaten declared innocent attempt regain honor kill moore k squeamish leave dirt path rain fall want warn not romance film dvd cover theatrical poster show indian woman caucasian man embrace idealized tropic set image capture directly film open quickly disappear seem like effort capitalize western fixation forbidden love not effective not m

In [26]:
errors["Actual"] = errors["Actual"].map({
    0: "Negative",
    1: "Positive"
})

errors["Predicted"] = errors["Predicted"].map({
    0: "Negative",
    1: "Positive"
})

errors.head(10)

,Review,Actual,Predicted
45455,not exactly great film admire writer director try something little different film main theme fate small seemingly insignificant thing greatly change future way remind film slide door though instead focus one random event seemingly random stuff happen repeatedly one help build cute conclusion plus odd bald guy seem understand talk one brief scene like sort omnipotent absolutely no explanation film like two guy fight clock tower hudsucker proxy dvd jacket show audrey tautou capitalize success amelie though one many actor film no one star role pace brisk acting fine conclusion not bad reason not score high character bit uninteresting think movie could perhaps tighten less subplots,Positive,Negative
34773,film start get feeling go something special acting camera work undoubtedly good also like character could grow empathise film good atmosphere hint fantasy however film go plot never appear takeoff roll scene scene unable understand connection story could see character occasionally bump reference ship bottle without connection leave unremarkable short story surprise well cannes,Negative,Positive
34520,despite huge fan fred astaire ginger rogers movie not year ago first saw follow fleet know song old astaire rogers record yes vinyl know nothing plot unfortunately song catchy ginger rogers character sweet funny not like bake baker try make longtime partner continually sabotage career character not usual humour elan film astaire character worth watch song great solo tap routine ginger rogers,Negative,Positive
27110,well well roeg touch bit nerve not genius catalogue various character descent psychosis couple decade soon bad taste suggest redemption even good advice might find bad old catholic church hip thou alternative movie crowd get extra vicious bad still theresa russell character face experience nothing avowedly rationalist outlook explanation unwillingly force deal experience another level spiritual know realm ignorant superstitious sort thing art house cinephiles suppose oh horror find marriage idea might uniquely important commitment affirm seem uncomfortably like divine intervention people find idea prima facie offensive could maybe ask instinctively jump attack mode challenge take seriously idea spiritual dimension life probably not sure film problem notably talia shire delirious hamwork overwrought nun style attire dialogue marie davenport young priest last scene straight spellbound school glib interpretation though hitchcock movie escape similar charge due source wisdom impeccably secular credential freudian psychoanalyst sadly nicolas roeg appear cop critical mauling much even ask question possible answer film present,Positive,Negative
33827,ok let admit not see merchant ivory distributor film nor see celebrated work director capacity discuss rain outside analysis film mitigate admittance let begin rain different kind movie not know genre want first pretend romance romance protagonist fall love support character separate support character sometimes united partner movie hero already win heart lover not name henry moore name sajani reside southern india waning day raj british imperial rule henry away london long time fall love married indian maid despite legal marriage child overseas could good often sneak away intimate afternoon child notice word spread sajani husband question involvement moore deny contact moor ask leave area sajani refuse devotion commits suicide please take note event occur opening third film film change tone become crime drama final portion sajani body discover right moore family come visit alleged perpetrator moore english educate assistant k k know master affair keep silent life becomes threaten declared innocent attempt regain honor kill moore k squeamish leave dirt path rain fall want warn not romance film dvd cover theatrical poster show indian woman caucasian man embrace idealized tropic set image capture directly film open quickly disappear seem like effort capita

In [27]:
error_summary = (
    errors
    .groupby(["Actual", "Predicted"])
    .size()
    .reset_index(name="Count")
)

error_summary

,Actual,Predicted,Count
0,Negative,Positive,589
1,Positive,Negative,464


In [28]:
decision_scores = svm_model.decision_function(X_test_tfidf)

error_analysis = pd.DataFrame({
    "Review": X_test,
    "Actual": y_test,
    "Predicted": y_pred_svm,
    "Decision Score": decision_scores
})

error_analysis = error_analysis[
    error_analysis["Actual"] != error_analysis["Predicted"]
].copy()

error_analysis["Confidence"] = error_analysis["Decision Score"].abs()

error_analysis = error_analysis.sort_values(
    by="Confidence",
    ascending=False
)

error_analysis.head(10)

,Review,Actual,Predicted,Decision Score,Confidence
43276,spoiler ahead really unfortunate movie well produce turn disappointment think full silly clichés basically try hard american guy many spend time jump girlfriend bed make monkey sound marry girl many suddenly go prude nymphos overnight not husband french would really ask someone à la fac know not speak french would not use common word like université live france sort know understand europe love german roommate find pretty insult overall look like movie fund european parliament try hard basically sort difference try tie together not bad thing result best awkward fact ridiculous many clash would not really happen end movie last minute ruin rest not xavier talk erasmus student meet back paris walk run away job freedom end new europe suppose rest bunch people smoke shag day make besides acting pretty horrible not believe judith godrèche role act make look like emanuelle béart much first think xavier ok retrospect think pretty bad really bad technically open credit scene ask paper need really good except sound edit around british sibling soundtrack great form good content pretty horrible,1,0,-1.866995,1.866995
23424,one best celebrity reality show ever saw see concert see life britney love five episode always surprise britney subject show think people not watch show great person really funny really gentle love fan see love work not give k fed real jerk not seem like britney make second season great show show people britney really go britney best never leave heart,0,1,1.775625,1.775625
22355,simply funny movie see long time bad act bad script bad scenery bad costume bad camera work bad special effect stupid find reel laughter not go win oscar get beer friend round not go wrong,1,0,-1.688947,1.688947
13773,comanche moon everything go starter simon wincer back man name synonymous high quality tv western unfortunately problem moon something even talented director could not solve poor script base lackluster novel forget historical accuracy reader novel tell big travesty moon not even consistent information original lonesome dove masterpiece many wonderful rich moment miniseries even great degree book completely miss moon consider fact viewer moon probably come least sort dove background lack character drive emotional backstory downright painful say moon one instance adapted version could alter make suitable screen example novel moon focus largely comanche credit miniseries try service comanche end give enough viewer get sense miss important part story similarly val kilmer scull lose role either expand kilmer rachel griffith matter actually something role reduce small support part instead kilmer get top billing character leave scratch head appearance completely bizarre final act strange moment throughout film make no sense not read book killer parrot not single scene show u call mccrae anything near amazing texas ranger claim not single one expedition moon dead man walk matter end successfully call mccrae seem blunder way one pointless mission another frankly larry mcmurtry never give job write script praise surround cowboy movie brokeback mountain mcmurtry write good novel although dissension consistency statement never exactly establish scriptwriter production would benefit not bring back director wincer original screenwriter bill wittcliff adapt novel screen witcliff not exactly mountainload material screen write credit no one deny fantastic job whittle original lonesome dove say comanche moon almost brilliant production aid terrific cast unfortunately not give enough steve zahn portrayal gus mccrae rather portrayal robert duvall gus mccrae dead criticize karl urban woodrow call say performance not imitate quiet stoic woodrow original movie say blame mcmurtry mcmurtry one moon novel miniseries turn woodrow socially inept awkward natural leader emotionless character whose line dull whose character motivation clear see portray far adequately sequel still cast sparkle moon degree le

In [29]:
least_confident_errors = error_analysis.sort_values(
    by="Confidence",
    ascending=True
)

least_confident_errors.head(10)

,Review,Actual,Predicted,Decision Score,Confidence
32880,slow film noir film ever see weak script provide opportunity jack palance rod steiger deliver exceptionally well line make acting audition palance steiger affirm indeed serious actor portray respective role power emotion drama story move incredibly slow pace mostly take place living room director obviously move actor around rythmatic circle deliver exhaustingly long line keep slow pace become noticeable audience action not pick last ten minute film one look depth crime drama one not,0,1,0.000525,0.000525
35232,matt cordell back dead third go round although not sure anyone care point except rabid manica cop fan cordell die last flick resurrect voodoo hot trail several miscreant involve shooting fellow officer cordell fond miss part early low budget quickie please see cordell wrack body count various gruesome way problem overall film pretty static cordell simply not jason freddy interest wan pretty fast even grand b movie master robert forster doctor end brain scramble stick first film series funny scary excite time,0,1,0.001155,0.001155
27844,yep read right kid michael bay study film make either hype overlong overly pointless transformer movie robot jox good probably cost less transformer crew spend megan fox personal trainer thankfully little robotic gem initially know mainly film bankrupt charles albert band empire picture studio seem develop cult follow year fondly remember watch vhs initial video release early though cold war era politics stereotype already date time band luck communism would fall film sit shelf wait release eh still pretty damn cool little b movie really not make like anymore go bay route cgi thing unbearable proportion unfamiliar robo scoop somewhere future nuclear holocaust large scale war outlaw dispute nation settle mano mano perhaps machine machine one representative side battle giant size shogun warrior style robot whichever bot walk away fight win side gary graham would later go play detective sykes alien nation tv series play achilles great robot jock marketplace k good guy history achilles undefeated previous nine robot bout ten maximum number battle jock retire beginning film face counterpart confederation e russkies psychotic alexander top evil russian stereotype bad guy since dolph lundgren infamous turn ivan drago rocky iv match call draw alexander violate rule illegal robot move last minute end not embarrass achilles kill whole bunch spectator bargain rematch schedule complete bout achilles simply want bow hang helmet move life rather violate spoiler rule reveal much simply say great deal twist turn behind scene skullduggery difficulty achilles fellow jox two robotic titan clash finally finale hope not make movie sort masterpiece science fiction not robot jox plain fun grant bit high concept average b grade sci fi movie though budgetary constraint occasionally make know especially scene involve painfully obvious green screen trickery still best look movie ever come empire full moon picture factory robot fight scene well use old school stop motion model technique set costume not look half assed slight empire picture director stuart animator gordon definitely shoot star picture unfortunately not quite pan studio least get one heckuva cool little movie deal bottom line want aurally visually assault hour feel free rent transformer movie end likely feel like spend time watch someone else play video game want rock sock robot good time pick robot jox instead,1,0,-0.002375,0.002375
31436,stone boy almost forgotten drama consider many famous soon famous people film one wonder could overlooked slow moody touch account tragedy befall farm family film less indictment midwestern stoic value suppression emotion film not taste anyone appreciate real human drama like ok early moment film see two brother head early morning hour pick pea maybe shoot duck two lucky climb barbed wire fence gun accidentally discharge young boy fatally shoot older brother bo

In [30]:
error_analysis.groupby(
    ["Actual", "Predicted"]
)["Confidence"].agg(
    ["count", "mean", "median", "min", "max"]
)

,,count,mean,median,min,max
Actual,Predicted,,,,,
0,1,589,0.288125,0.229179,0.000525,1.775625
1,0,464,0.281874,0.199819,0.002375,1.866995


In [31]:
# 1. Define your custom text review
custom_review = ["Definite masterpiece if your goal is to fall asleep in five minutes."]

# 2. Transform the text using your loaded TF-IDF Vectorizer
custom_tfidf = tfidf.transform(custom_review)

# 3. Compute the raw Decision Score (distance from the SVM hyperplane)
custom_score = svm_model.decision_function(custom_tfidf)[0]

# 4. Generate the final hard class prediction (0 or 1)
custom_pred = svm_model.predict(custom_tfidf)[0]
predicted_label = "Positive" if custom_pred == 1 else "Negative"

# 5. Output the results
print(f"Review:         '{custom_review[0]}'")
print(f"Decision Score:  {custom_score:.4f} (Positive value = Positive Sentiment)")
print(f"Prediction:      {predicted_label} ({custom_pred})")

Review:         'Definite masterpiece if your goal is to fall asleep in five minutes.'
Decision Score:  -0.0672 (Positive value = Positive Sentiment)
Prediction:      Negative (0)


In [32]:
# 1. Isolate the specific review tokens
review_text = "The visual effects were beautiful and spectacular, but the story was absolute garbage."
transformed = tfidf.transform([review_text])

# 2. Extract words that actually triggered a match (non-zero TF-IDF values)
matched_indices = transformed.nonzero()[1]
feature_names = tfidf.get_feature_names_out()

# 3. Pair the active tokens with their learned SVM weights
token_weights = []
for idx in matched_indices:
    word = feature_names[idx]
    weight = svm_model.coef_[0][idx]
    tfidf_val = transformed[0, idx]
    token_weights.append({"Token": word, "SVM_Weight": weight, "TF-IDF_Value": tfidf_val})

# 4. Display the calculation breakdown
breakdown_df = pd.DataFrame(token_weights).sort_values(by="SVM_Weight")
print("=== Mathematical Breakdown of the Review ===")
print(breakdown_df.to_string(index=False))

=== Mathematical Breakdown of the Review ===
      Token  SVM_Weight  TF-IDF_Value
    garbage   -0.848514      0.446193
   absolute    0.033493      0.472361
      story    0.145730      0.190469
     visual    0.174560      0.436177
spectacular    0.472709      0.497127
  beautiful    0.887657      0.322682
